# ANÁLISIS DE SENTIMIENTOS CON REDES LSTM

## 1 - El problema a resolver

Crearemos un modelo LSTM que sea capaz de clasificar una crítica a una película (en formato texto) como positiva o negativa.

##2 - El set de datos


Usaremos el dataset [IMDB](https://ai.stanford.edu/~amaas/data/sentiment/) (*Internet Movie Database*) que contiene 50.000 críticas a diferentes películas, etiquetadas por sentimiento: 0 para negativos y 1 para positivos.

Algunos ejemplos:

- *a rating of 1 does not begin to express how dull depressing and relentlessly bad this movie is...*. Sentimiento: **NEGATIVO**
- *this film was just brilliant casting location scenery story direction everyone's really suited the part they played and you could just imagine being there robert redford's is an amazing actor...* Sentimiento: **POSITIVO**

In [1]:
# Leer el dataset

from tensorflow.keras.datasets import imdb

# Antes de leer debemos tener en cuenta que habrá:
# - Espacios en blanco: índice 0
# - Token (indicador) de inicio de frase: índice 1
# - Token para indicar palabra desconocida: índice 2
# - Token para indicar palabra no usada: índice 3

(x_train, y_train), (x_test, y_test) = imdb.load_data(index_from=3)

17464789/17464789 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


In [2]:
print(x_train.shape, y_train.shape)
print(x_test.shape, y_test.shape)

(25000,) (25000,)
(25000,) (25000,)


In [3]:
# Cada palabra en un review está representada con un número entero,
# que indica la frecuencia con que la palabra aparece en el dataset
# Por ejemplo, "10" indica que es la décima palabra más común en el
# dataset

print(x_train[0])

[1, 14, 22, 16, 43, 530, 973, 1622, 1385, 65, 458, 4468, 66, 3941, 4, 173, 36, 256, 5, 25, 100, 43, 838, 112, 50, 670, 22665, 9, 35, 480, 284, 5, 150, 4, 172, 112, 167, 21631, 336, 385, 39, 4, 172, 4536, 1111, 17, 546, 38, 13, 447, 4, 192, 50, 16, 6, 147, 2025, 19, 14, 22, 4, 1920, 4613, 469, 4, 22, 71, 87, 12, 16, 43, 530, 38, 76, 15, 13, 1247, 4, 22, 17, 515, 17, 12, 16, 626, 18, 19193, 5, 62, 386, 12, 8, 316, 8, 106, 5, 4, 2223, 5244, 16, 480, 66, 3785, 33, 4, 130, 12, 16, 38, 619, 5, 25, 124, 51, 36, 135, 48, 25, 1415, 33, 6, 22, 12, 215, 28, 77, 52, 5, 14, 407, 16, 82, 10311, 8, 4, 107, 117, 5952, 15, 256, 4, 31050, 7, 3766, 5, 723, 36, 71, 43, 530, 476, 26, 400, 317, 46, 7, 4, 12118, 1029, 13, 104, 88, 4, 381, 15, 297, 98, 32, 2071, 56, 26, 141, 6, 194, 7486, 18, 4, 226, 22, 21, 134, 476, 26, 480, 5, 144, 30, 5535, 18, 51, 36, 28, 224, 92, 25, 104, 4, 226, 65, 16, 38, 1334, 88, 12, 16, 283, 5, 16, 4472, 113, 103, 32, 15, 16, 5345, 19, 178, 32]


In [4]:
# Cada crítica tendrá un tamaño diferente, así que tendremos
# que ajustar todas las secuencias al mismo tamaño
print(len(x_train[0]))
print(len(x_train[200]))

218
160


In [5]:
# Tamaño del vocabulario: ¿cuántas palabras diferentes aparecen
# en el dataset
import numpy as np

TAM_VOCAB = len(np.unique(x_train))
print(f"Tamaño del vocabulario: {TAM_VOCAB} palabras")

Tamaño del vocabulario: 24902 palabras


In [6]:
# Convertir el "review" de su representación numérica a texto

palabra_to_id = imdb.get_word_index()
palabra_to_id = {key:(value+3) for key,value in palabra_to_id.items()}
palabra_to_id["<PAD>"] = 0
palabra_to_id["<START>"] = 1
palabra_to_id["<UNK>"] = 2
palabra_to_id["<UNUSED>"] = 3

# Diccionario inverso: de índices a palabras
id_to_palabra = dict((i, palabra) for (palabra, i) in palabra_to_id.items())

print(palabra_to_id['where'])
print(id_to_palabra[25])

1641221/1641221 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
121
you


In [7]:
# Ejemplo de una frase del set de entrenamiento
import textwrap
ID = 159
texto_decodificado = " ".join(id_to_palabra[i] for i in x_train[ID])
print('REVIEW:')
print(textwrap.fill(texto_decodificado,80)+'\n')
print('SENTIMIENTO:')

if y_train[ID]==0:
  print('Negativo')
else:
  print('Positivo')

REVIEW:
<START> a rating of 1 does not begin to express how dull depressing and
relentlessly bad this movie is

SENTIMIENTO:
Negativo


## 3 - El sistema a implementar

![](https://drive.google.com/uc?export=view&id=1xRRRsllKQPim61HFMKiOiVDYyfFDh8w8)

## 4 - Padding con ceros

Las entradas deben tener un tamaño fijo, pero cada secuencia (texto con el sentimiento) tiene un tamaño variable.

Fijaremos el tamaño de cada *review* a 100 palabras:

- Si la secuencia tiene menos de 100 palabras, el resto de los elementos se completará con ceros
- Se truncarán secuencias con más de 100 palabras

In [8]:
from tensorflow.keras.preprocessing import sequence

MAX_LEN = 100 # Máxima longitud de la secuencia
x_train_padded = sequence.pad_sequences(x_train, maxlen=MAX_LEN, padding='post',
                                        truncating='post')
x_test_padded = sequence.pad_sequences(x_test, maxlen=MAX_LEN, padding='post',
                                        truncating='post')
print(x_train.shape)
print(x_train_padded.shape)

(25000,)
(25000, 100)


In [9]:
print(x_train[0])

[1, 14, 22, 16, 43, 530, 973, 1622, 1385, 65, 458, 4468, 66, 3941, 4, 173, 36, 256, 5, 25, 100, 43, 838, 112, 50, 670, 22665, 9, 35, 480, 284, 5, 150, 4, 172, 112, 167, 21631, 336, 385, 39, 4, 172, 4536, 1111, 17, 546, 38, 13, 447, 4, 192, 50, 16, 6, 147, 2025, 19, 14, 22, 4, 1920, 4613, 469, 4, 22, 71, 87, 12, 16, 43, 530, 38, 76, 15, 13, 1247, 4, 22, 17, 515, 17, 12, 16, 626, 18, 19193, 5, 62, 386, 12, 8, 316, 8, 106, 5, 4, 2223, 5244, 16, 480, 66, 3785, 33, 4, 130, 12, 16, 38, 619, 5, 25, 124, 51, 36, 135, 48, 25, 1415, 33, 6, 22, 12, 215, 28, 77, 52, 5, 14, 407, 16, 82, 10311, 8, 4, 107, 117, 5952, 15, 256, 4, 31050, 7, 3766, 5, 723, 36, 71, 43, 530, 476, 26, 400, 317, 46, 7, 4, 12118, 1029, 13, 104, 88, 4, 381, 15, 297, 98, 32, 2071, 56, 26, 141, 6, 194, 7486, 18, 4, 226, 22, 21, 134, 476, 26, 480, 5, 144, 30, 5535, 18, 51, 36, 28, 224, 92, 25, 104, 4, 226, 65, 16, 38, 1334, 88, 12, 16, 283, 5, 16, 4472, 113, 103, 32, 15, 16, 5345, 19, 178, 32]


In [10]:
print(x_train_padded[0,:])

[    1    14    22    16    43   530   973  1622  1385    65   458  4468
    66  3941     4   173    36   256     5    25   100    43   838   112
    50   670 22665     9    35   480   284     5   150     4   172   112
   167 21631   336   385    39     4   172  4536  1111    17   546    38
    13   447     4   192    50    16     6   147  2025    19    14    22
     4  1920  4613   469     4    22    71    87    12    16    43   530
    38    76    15    13  1247     4    22    17   515    17    12    16
   626    18 19193     5    62   386    12     8   316     8   106     5
     4  2223  5244    16]


## 5 - Word embeddings

En lugar de representar cada palabra como un vector one-hot de 24902 elementos (el tamaño del vocabulario), resulta más eficiente entrenar una pequeña Red Neuronal que represente de forma compacta (128 elementos) cada palabra.

Esta representación compacta se conoce como **embedding** y palabras similares tendrán *embeddings* similares:

![](https://drive.google.com/uc?export=view&id=1YjgUt4VF_bFkzADjdr-gABkKXHUOVub-)

In [11]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding
import tensorflow as tf

# Semilla de los generadores aleatorios
SEED = 32
tf.random.set_seed(SEED)
np.random.seed(SEED)

# Parámetros del embedding
INPUT_DIM = TAM_VOCAB    # El tamaño del vocabulario
OUTPUT_DIM = 128         # Tamaño de cada vector codificado por el word embedding
INPUT_LENGTH = 100       # La longitud (número de elementos) de cada secuencia

# Creación del modelo y adición del embedding
modelo = Sequential()
modelo.add(Embedding(input_dim=INPUT_DIM, output_dim=OUTPUT_DIM,
                     input_length=INPUT_LENGTH))

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/embedding.py:100: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


## 6 - El modelo: red LSTM y capa de salida

In [12]:
from tensorflow.keras.layers import LSTM, Dense

# 1. Reducimos la capacidad de la memoria LSTM
UNITS = 32

# Red LSTM
modelo = Sequential()
modelo.add(Embedding(input_dim=INPUT_DIM, output_dim=OUTPUT_DIM, input_length=INPUT_LENGTH))
modelo.add(LSTM(UNITS))

# Capa de salida (clasificación de sentimientos)
modelo.add(Dense(1, activation='sigmoid'))

### Número de parámetros del modelo:

#### Embedding

- El dato de entrada será del mismo tamaño del vocabulario (24.902 elementos) y la salida será de 128. Para esta capa se deben entrenar 24902x128 = 3'187.456 parámetros

#### Red LSTM

- Las compuertas "olvidar", "entrada", "salida" y "candidata" son pequeñas redes neuronales cuya salida es de la forma:
$f\left( \bar{W_x}\bar{x_t} + \bar{W_h}\bar{h_{t-1}} + \bar{b} \right)$
- En este caso:
  - $\bar{x_t}$ (dato de entrada): 128x1 (pues el *embedding* genera vectores de 128 elementos)
  - $\bar{h_{t-1}}$, $\bar{c_t}$ (estado oculto, celda de memoria): 128x1 (pues la variable ``UNITS`` de la red LSTM es 128)
- Los coeficientes de cada compuerta y de la celda candidata tendrán estos tamaños:
  - $\bar{W_x}$: 128x128 = 16.384 parámetros
  - $\bar{W_h}$: 128x128 = 16.384 parámetros
  - $\bar{b}$: 128x1  = 128 parámetros
- En total cada elemento requiere el entrenamiento de 16.384 + 16.384 + 128 = 32.896 parámetros.
- Al tener cuatro de estos elementos se requerirán 4x32.896 = 131.584 parámetros

#### Capa de salida:
- El dato de entrada será de 128 elementos (la salida de la red LSTM) y a la salida se tendrá una única neurona (clasificación softmax).
- El número total de parámetros será: 128x1 (Ws) + 1 (b) = 129

In [13]:
modelo.summary()

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_1 (Embedding)         │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm (LSTM)                     │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

## 7 - Entrenamiento del modelo

In [14]:
from tensorflow.keras.optimizers import RMSprop

# 2. Aumentamos ligeramente la tasa de aprendizaje para compensar menos épocas
optimizador = RMSprop(learning_rate = 0.0005)
modelo.compile(loss='binary_crossentropy', optimizer=optimizador, metrics=['accuracy'])

# 3. Reducimos el Batch Size (más ruido regularizador)
BATCH_SIZE = 64

# 4. Reducimos las iteraciones para evitar que empiece a memorizar (Early Stopping manual)
EPOCHS = 5

historia = modelo.fit(x_train_padded, y_train,
                      batch_size = BATCH_SIZE,
                      epochs = EPOCHS,
                      validation_data = (x_test_padded, y_test))

Epoch 1/5
391/391 ━━━━━━━━━━━━━━━━━━━━ 10s 15ms/step - accuracy: 0.6516 - loss: 0.6014 - val_accuracy: 0.7460 - val_loss: 0.5457
Epoch 2/5
391/391 ━━━━━━━━━━━━━━━━━━━━ 5s 12ms/step - accuracy: 0.8270 - loss: 0.4029 - val_accuracy: 0.8100 - val_loss: 0.4409
Epoch 3/5
391/391 ━━━━━━━━━━━━━━━━━━━━ 5s 14ms/step - accuracy: 0.8716 - loss: 0.3248 - val_accuracy: 0.8128 - val_loss: 0.4533
Epoch 4/5
391/391 ━━━━━━━━━━━━━━━━━━━━ 5s 13ms/step - accuracy: 0.8938 - loss: 0.2796 - val_accuracy: 0.8107 - val_loss: 0.4685
Epoch 5/5
391/391 ━━━━━━━━━━━━━━━━━━━━ 5s 13ms/step - accuracy: 0.9108 - loss: 0.2440 - val_accuracy: 0.8102 - val_loss: 0.4848


El modelo tiene **overfitting**, pues se alcanza una exactitud del 90% con el set de entrenamiento y del 80% con el de prueba.

Sugerencia: refinar los hiper-parámetros del modelo:
- Tamaño fijo de las secuencias (originalmente de 100 elementos)
- Tamaño del *embedding* (originalmente 128 elementos)
- Número de unidades de la red LSTM (originalmente 128)
- Tasa de aprendizaje del optimizador RMSprop (originalmente 0.00005)
- Batch size (originalmente 128)
- Número de iteraciones de entrenamiento (originalmente 10)

En todo caso veamos qué tan bien lo hace este modelo.

## 8 - Predicción con el modelo entrenado

In [15]:
# Tomemos un dato de prueba y verifiquemos si lo clasifica correctamente
# Podemos probar con el dato 25 (POSITIVO), el 5 (POSITIVO) o el 15 (NEGATIVO),
# por ejemplo
ID = 5
texto_decodificado = " ".join(id_to_palabra[i] for i in x_test_padded[ID])
print(textwrap.fill(texto_decodificado,80))

<START> i'm absolutely disgusted this movie isn't being sold all who love this
movie should email disney and increase the demand for it they'd eventually have
to sell it then i'd buy copies for everybody i know everything and everybody in
this movie did a good job and i haven't figured out why disney hasn't put this
movie on dvd or on vhs in rental stores at least i haven't seen any copies this
is a wicked good movie and should be seen by all the kids in the new generation
don't get to see it and i think they


In [16]:
# Predicción (usar x_test_padded)
prob = modelo.predict(x_test_padded[ID,:].reshape(1,100,1))
if prob<=0.5:
  print('Sentimiento: NEGATIVO')
else:
  print('Sentimiento: POSITIVO')

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 127ms/step
Sentimiento: POSITIVO
